# Scraping de datos de médicos y clínicas disponibles

In [2]:
import requests
import pandas as pd
import math
import time
from datetime import datetime, timedelta
import urllib3

In [7]:
# Desactivamos las alertas de seguridad SSL para que no ensucien la consola

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIGURACIÓN ---
BASE_URL = 'https://app9.susalud.gob.pe:8089/api/renam-consulta/consulta/consultaProgramacion'

# TUS HEADERS EXACTOS (Copiados de tu input)
headers = {
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'es-ES,es;q=0.9',
    'Connection': 'keep-alive',
    'Origin': 'https://tua.susalud.gob.pe:8084',
    'Referer': 'https://tua.susalud.gob.pe:8084/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36',
    'sec-ch-ua': '"Chromium";v="142", "Google Chrome";v="142", "Not_A Brand";v="99"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    # Este header parece ser el token de seguridad dinámico:
    'transfer-encoding-vary-header': 'Vnh4XU9ff0BoaWRGamUAR1MAf0JS', 
}

# TUS PARÁMETROS INICIALES
# Nota: Profesion 7 = Biólogos (según tu JSON anterior). 
# Para médicos, cambiarás esto a '1' más adelante.
params = {
    'feInicio': '2025-11-21',
    'feFin': '2025-12-20',
    'profesion': '1', 
    'especialidad': '84',
    'nombreIpress': '',
    'nombre': '',
    'paterno': '',
    'materno': '',
    'ubigeo': '15', # Lima
    'size': '10',
    'page': '0', # Empezamos en la página 0
}

def extraer_turnos_susalud():
    print("🚀 Iniciando extracción con tus credenciales...")
    
    all_data = []
    
    # 1. PRIMERA PETICIÓN (Para saber cuántas páginas son)
    try:
        response = requests.get(BASE_URL, params=params, headers=headers, verify=False)
        data = response.json()
        
        if not data.get("success"):
            print("❌ Error en la respuesta de la API:", data)
            return
            
        total_registros = data["total"]
        # Calculamos páginas (Total / 10 items por página)
        total_paginas = math.ceil(total_registros / 10)
        
        print(f"✅ Conexión exitosa. Se encontraron {total_registros} registros en {total_paginas} páginas.")
        
    except Exception as e:
        print(f"❌ Error fatal conectando: {e}")
        return

    # 2. LOOP DE EXTRACCIÓN (Página por página)
    for pagina in range(total_paginas):
        print(f"📄 Procesando página {pagina + 1} de {total_paginas}...", end="\r")
        
        # Actualizamos el número de página en los parámetros
        params["page"] = str(pagina)
        
        try:
            # Hacemos la petición
            resp = requests.get(BASE_URL, params=params, headers=headers, verify=False)
            page_data = resp.json()
            
            if page_data.get("data"):
                all_data.extend(page_data["data"])
            
            # Pausa pequeña para no saturar (buena práctica en scraping)
            time.sleep(0.2)
            
        except Exception as e:
            print(f"\n⚠️ Falló la página {pagina}: {e}")

    print(f"\n✨ ¡Extracción completa! {len(all_data)} registros descargados.")
    
    # 3. GUARDAR RESULTADOS
    if all_data:
        # Aplanamos el JSON para que quede bonito en Excel
        df = pd.json_normalize(all_data)
        
        
        filename = "susalud_endocrinologos_lima.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"💾 Archivo guardado exitosamente: {filename}")
        print(df.head())
    else:
        print("⚠️ No se encontraron datos para guardar.")

if __name__ == "__main__":
    extraer_turnos_susalud()

🚀 Iniciando extracción con tus credenciales...
✅ Conexión exitosa. Se encontraron 1301 registros en 131 páginas.
📄 Procesando página 131 de 131...
✨ ¡Extracción completa! 1301 registros descargados.
💾 Archivo guardado exitosamente: susalud_endocrinologos_lima.csv
  institucion                  establecimiento  \
0     PUBLICO  HOSPITAL SAN JUAN DE LURIGANCHO   
1     PUBLICO  HOSPITAL SAN JUAN DE LURIGANCHO   
2     PUBLICO  HOSPITAL SAN JUAN DE LURIGANCHO   
3     PUBLICO  HOSPITAL SAN JUAN DE LURIGANCHO   
4     PUBLICO    CENTRO MATERNO INFANTIL RÍMAC   

                                           actividad codigo_Unico  \
0  CE : CONSULTA MÉDICA AMBULATORIA (MÉDICO GENER...     00005617   
1  CE : CONSULTA MÉDICA AMBULATORIA (MÉDICO GENER...     00005617   
2  CE : CONSULTA MÉDICA AMBULATORIA (MÉDICO GENER...     00005617   
3  CE : CONSULTA MÉDICA AMBULATORIA (MÉDICO GENER...     00005617   
4  CE : CONSULTA MÉDICA AMBULATORIA (MÉDICO GENER...     00005644   

                    